In [1]:
# 06_impute_features
#
# 목적: model_table.csv의 결측치를 최종적으로 채운다. 두 단계로 나눠서 처리한다.
#
# [A단계 - 결정적 규칙, 분할과 무관하게 안전]
#   "해당 소스 테이블에 이 유저가 아예 없다" -> count/day/sum류 컬럼은 값 자체가 0이 맞음
#   (예: user_logs에 한 번도 로그가 없으면 active_days=0이 되는 게 자연스러움).
#   이런 컬럼은 0으로 채우고, 대신 "원래 그 소스에 존재했는지" 여부를 플래그 컬럼으로 남겨
#   정보 손실을 방지한다 (has_transaction_record, has_log_activity).
#
# [B단계 - 통계 기반, 반드시 train split으로만 학습]
#   평균/비율류 컬럼(예: cancel_rate, d30_avg_total_secs)은 0으로 채우면 의미가 왜곡된다
#   (활동이 없어서 못 구한 것과 실제로 0인 것은 다름). 이런 컬럼은 train split의 중앙값으로
#   대체하고, 그 대체값을 valid/test에도 동일하게 적용한다 (train 통계만 사용 -> 리키지 방지).
#
# 스케일링(정규화)은 여기서 하지 않는다 -> 트리 모델은 불필요하고, 필요한 모델(DL/선형)은
# 모델링 단계에서 train으로 fit해서 적용하는 것이 더 적절하므로 이 공용 테이블에는 포함하지 않는다.
#
# 입력: data/processed/model_table.csv
# 출력:
#   data/processed/model_table_final.csv   (결측 없는 최종 모델링 테이블)
#   data/processed/impute_values.json      (train에서 계산한 대체값 기록, 재현성용)

In [2]:
import json
import pandas as pd
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
table = pd.read_csv(PROCESSED_DIR / "model_table.csv")
print(table.shape)

(992931, 41)


In [3]:
# A단계: 존재 플래그 생성 (원본 결측 여부를 정보로 보존)
table["has_member_record"] = table["city"].notna().astype("int8")
table["has_transaction_record"] = table["txn_count"].notna().astype("int8")
table["has_log_activity"] = table["full_active_days"].notna().astype("int8")

# A단계: count/day/sum류 컬럼 -> 0으로 채움 (해당 소스에 유저가 아예 없으면 0이 맞는 값)
ZERO_FILL_COLS = [
    "txn_count", "payment_method_nunique", "plan_days_nunique", "cancel_count", "total_amount_paid",
    "d7_active_days", "d30_active_days", "d90_active_days", "full_active_days",
]
table[ZERO_FILL_COLS] = table[ZERO_FILL_COLS].fillna(0)

# A단계: members 범주형 컬럼 -> 결정적 규칙으로 채움 (통계 아님, 그냥 "정보 없음" 표시)
# city/registered_via는 실제 데이터에 -1 같은 코드가 이미 쓰이고 있어서 겹치지 않는 sentinel 사용
MISSING_CATEGORY_SENTINEL = -999
table["city"] = table["city"].fillna(MISSING_CATEGORY_SENTINEL)
table["registered_via"] = table["registered_via"].fillna(MISSING_CATEGORY_SENTINEL)
table["gender"] = table["gender"].fillna("unknown")
table["bd_is_missing"] = table["bd_is_missing"].fillna(1).astype("int8")

print("A단계 후 has_member_record 비율:", table["has_member_record"].mean())
print("A단계 후 has_transaction_record 비율:", table["has_transaction_record"].mean())
print("A단계 후 has_log_activity 비율:", table["has_log_activity"].mean())

A단계 후 has_member_record 비율: 0.8834057955688764
A단계 후 has_transaction_record 비율: 1.0
A단계 후 has_log_activity 비율: 0.8761192872415102


In [4]:
remaining_missing = table.isna().sum()
remaining_missing = remaining_missing[remaining_missing > 0].sort_values(ascending=False)
print(f"A단계 이후에도 결측이 남은 컬럼 ({len(remaining_missing)}개):")
print(remaining_missing)

A단계 이후에도 결측이 남은 컬럼 (16개):
bd_clean                          603791
d7_avg_total_secs                 325401
d7_completion_rate                325394
d7_avg_num_unq                    325394
trend_secs_recent15_vs_prior15    301259
d30_avg_total_secs                225771
d30_completion_rate               225763
d30_avg_num_unq                   225763
d90_avg_total_secs                183811
d90_avg_num_unq                   183808
d90_completion_rate               183808
full_avg_num_unq                  123005
full_avg_total_secs               123005
full_completion_rate              123005
tenure_days                       115770
days_to_expire                        82
dtype: int64


In [5]:
# B단계: train split의 중앙값으로만 학습해서 대체 (valid/test는 transform만)
MEDIAN_FILL_COLS = remaining_missing.index.tolist()

train_medians = table.loc[table["split"] == "train", MEDIAN_FILL_COLS].median()
print("train 중앙값:")
print(train_medians)

table[MEDIAN_FILL_COLS] = table[MEDIAN_FILL_COLS].fillna(train_medians)

train 중앙값:
bd_clean                            28.000000
d7_avg_total_secs                 4793.050800
d7_completion_rate                   0.756098
d7_avg_num_unq                      19.600000
trend_secs_recent15_vs_prior15      -0.001817
d30_avg_total_secs                4784.615227
d30_completion_rate                  0.739018
d30_avg_num_unq                     19.692308
d90_avg_total_secs                4978.109334
d90_avg_num_unq                     20.360000
d90_completion_rate                  0.728571
full_avg_num_unq                    21.305320
full_avg_total_secs               5231.090009
full_completion_rate                 0.715848
tenure_days                       1021.000000
days_to_expire                      15.000000
dtype: float64


In [6]:
final_missing = table.isna().sum().sum()
assert final_missing == 0, f"결측치가 남아있음: {final_missing}"
print("결측치 0건 확인 완료")
print(table.shape)

결측치 0건 확인 완료
(992931, 44)


In [7]:
table.to_csv(PROCESSED_DIR / "model_table_final.csv", index=False)

with open(PROCESSED_DIR / "impute_values.json", "w", encoding="utf-8") as f:
    json.dump(train_medians.to_dict(), f, ensure_ascii=False, indent=2)

print(f"저장 완료: {PROCESSED_DIR / 'model_table_final.csv'} ({len(table):,} rows, {len(table.columns)} columns)")
print(f"저장 완료: {PROCESSED_DIR / 'impute_values.json'}")

저장 완료: ..\data\processed\model_table_final.csv (992,931 rows, 44 columns)
저장 완료: ..\data\processed\impute_values.json
